<a href="https://colab.research.google.com/github/Akshad628/hyderabad-urban-heat-analytics/blob/main/Hyderabad_Urban_Heat_Island_Big_Data_Analytics_cloud_computing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!apt-get update -qq
!apt-get install -y -qq openjdk-11-jdk-headless
!java -version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.32.1+1-1ubuntu1~24.04_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.32.1+1-1ubuntu1~24.04) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.32.1+1-1ubuntu1~24.04_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.32.1+1-1ubuntu1~24.04) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.32.1+1-1ubuntu1~24.04) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provide /us

In [3]:

!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.4.1/hadoop-3.4.1.tar.gz -O /content/hadoop.tar.gz
!tar -xzf /content/hadoop.tar.gz -C /content/
!ln -sfn /content/hadoop-3.4.1 /content/hadoop
!echo 'export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64' >> /content/hadoop/etc/hadoop/hadoop-env.sh
!export HADOOP_HOME=/content/hadoop
!export PATH=$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH
!/content/hadoop/bin/hadoop version


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.4.1
Source code repository https://github.com/apache/hadoop.git -r 4d7825309348956336b8f06a08322b78422849b1
Compiled by mthakur on 2024-10-09T14:57Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum 7292fe9dba5e2e44e3a9f763fce3e680
This command was run using /content/hadoop-3.4.1/share/hadoop/common/hadoop-common-3.4.1.jar


In [5]:
import os
import pandas as pd

for f in ["hyderabad_heat.csv", "mapper.py", "reducer.py"]:
    print(f, "—", os.path.getsize(f"{'/content/'}{f}"), "bytes")

df = pd.read_csv("/content/hyderabad_heat.csv")
print("\nCSV shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

hyderabad_heat.csv — 9268376 bytes
mapper.py — 972 bytes
reducer.py — 1326 bytes

CSV shape: (95436, 6)
Columns: ['grid_id', 'year', 'latitude', 'longitude', 'lst_c', 'ndvi']


,grid_id,year,latitude,longitude,lst_c,ndvi
0,78.64422_17.19861,2022,17.198609,78.644219,41.957388,0.446241
1,78.64704_17.19864,2022,17.198642,78.647038,44.382379,0.298332
2,78.64986_17.19867,2022,17.198675,78.649857,43.401723,0.289575
3,78.43277_17.19874,2022,17.198735,78.432775,46.693593,0.239513
4,78.43559_17.19877,2022,17.198771,78.435594,46.489185,0.265114


In [6]:

from pathlib import Path
import pandas as pd

base = Path("/content")

# Inspect the exact code before changing anything
for name in ("mapper.py", "reducer.py"):
    path = base / name
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print(path.read_text())

# Verify the dataset structure and aggregation requirements
df = pd.read_csv(base / "hyderabad_heat.csv")

print("\n========== DATA AUDIT ==========")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("Missing values:\n", df.isna().sum().to_string())
print("Invalid temperature values:", (~df["lst_c"].between(-50, 80)).sum())
print("Invalid NDVI values:", (~df["ndvi"].between(-1, 1)).sum())
print("Duplicate grid/year records:",
      df.duplicated(["grid_id", "year"]).sum())
print("\nRecords by year:\n", df.groupby("year").size().to_string())



==================== mapper.py ====================

#!/usr/bin/env python3
import sys
import csv

reader = csv.reader(sys.stdin)

for row in reader:
    if len(row) < 6:
        continue

    if row[0].strip().lower() == "grid_id":
        continue

    try:
        grid_id = row[0].strip()
        year = int(row[1].strip())
        latitude = float(row[2])
        longitude = float(row[3])
        temperature = float(row[4])
        ndvi = float(row[5])

        if not grid_id or not (2022 <= year <= 2025):
            continue
        if not (-90 <= temperature <= 80):
            continue
        if not (-1.1 <= ndvi <= 1.1):
            continue
        if not (-90 <= latitude <= 90):
            continue
        if not (-180 <= longitude <= 180):
            continue

        key = f"{year}|{grid_id}"
        print(
            f"{key}\t"
            f"{temperature},{ndvi},{latitude},{longitude},1"
        )

    except (ValueError, IndexError):
        continue


==============

In [7]:

from pathlib import Path
import os
import subprocess
import pandas as pd

BASE = Path("/content")
INPUT = BASE / "hyderabad_heat.csv"
MAPPER = BASE / "mapper.py"
REDUCER = BASE / "reducer.py"
OUTPUT = BASE / "uhi_mapreduce_output"

# Fail early if the expected files or Hadoop installation are missing.
assert INPUT.is_file(), f"Missing input CSV: {INPUT}"
assert Path("/content/hadoop/bin/hadoop").is_file(), "Hadoop executable missing"

# Mapper: group each record by year and rounded geographic coordinates.
MAPPER.write_text(r'''#!/usr/bin/env python3
import sys, csv

reader = csv.reader(sys.stdin)
for row in reader:
    if len(row) < 6 or row[0].strip().lower() == "grid_id":
        continue
    try:
        grid_id = row[0].strip()
        year = int(row[1])
        lat, lon, temp, ndvi = map(float, row[2:6])

        if not grid_id or year not in (2022, 2023, 2024, 2025):
            continue
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            continue
        if not (-50 <= temp <= 80 and -1 <= ndvi <= 1):
            continue

        zone_lat = round(lat, 2)
        zone_lon = round(lon, 2)
        key = f"{year}|{zone_lat:.2f}|{zone_lon:.2f}"
        print(f"{key}\t{temp},{ndvi},{lat},{lon},1")
    except (ValueError, IndexError):
        continue
''')

# Reducer: calculate actual statistics for every year/location group.
REDUCER.write_text(r'''#!/usr/bin/env python3
import sys

current_key = None
count = 0
temp_sum = ndvi_sum = lat_sum = lon_sum = 0.0
temp_min = float("inf")
temp_max = float("-inf")

def emit(key):
    if key is None or count == 0:
        return
    year, zone_lat, zone_lon = key.split("|")
    print(
        f"{year}\t{zone_lat}\t{zone_lon}\t"
        f"{temp_sum/count:.3f}\t{ndvi_sum/count:.5f}\t"
        f"{count}\t{temp_min:.3f}\t{temp_max:.3f}\t"
        f"{lat_sum/count:.6f}\t{lon_sum/count:.6f}"
    )

for line in sys.stdin:
    try:
        key, values = line.rstrip("\n").split("\t", 1)
        temp, ndvi, lat, lon, number = values.split(",")
        temp, ndvi, lat, lon = map(float, (temp, ndvi, lat, lon))
        number = int(number)
        if number <= 0:
            continue
    except (ValueError, IndexError):
        continue

    if key != current_key:
        emit(current_key)
        current_key = key
        count = 0
        temp_sum = ndvi_sum = lat_sum = lon_sum = 0.0
        temp_min = float("inf")
        temp_max = float("-inf")

    temp_sum += temp * number
    ndvi_sum += ndvi * number
    lat_sum += lat * number
    lon_sum += lon * number
    count += number
    temp_min = min(temp_min, temp)
    temp_max = max(temp_max, temp)

emit(current_key)
''')

# Locate the installed Hadoop Streaming JAR without downloading anything.
jars = list(Path("/content/hadoop/share/hadoop/tools/lib").glob(
    "hadoop-streaming-*.jar"
))
assert len(jars) == 1, f"Expected one Streaming JAR; found: {jars}"

# Validate the input schema before launching Hadoop.
df = pd.read_csv(INPUT)
expected = ["grid_id", "year", "latitude", "longitude", "lst_c", "ndvi"]
assert list(df.columns) == expected, f"Unexpected CSV columns: {df.columns.tolist()}"
assert len(df) == 95436, f"Unexpected row count: {len(df)}"

# Remove only the previous output directory created by this notebook.
if OUTPUT.exists():
    import shutil
    shutil.rmtree(OUTPUT)

env = os.environ.copy()
env["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
env["HADOOP_HOME"] = "/content/hadoop"
env["PATH"] = f"/content/hadoop/bin:{env.get('PATH', '')}"

cmd = [
    "/content/hadoop/bin/hadoop", "jar", str(jars[0]),
    "-D", "mapreduce.framework.name=local",
    "-D", "fs.defaultFS=file:///",
    "-D", "mapreduce.job.reduces=1",
    "-input", str(INPUT),
    "-output", str(OUTPUT),
    "-mapper", f"python3 {MAPPER}",
    "-reducer", f"python3 {REDUCER}",
]

print("Starting Hadoop Streaming job...")
result = subprocess.run(
    cmd, env=env, text=True, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

if result.returncode != 0:
    print(result.stdout[-12000:])
    raise RuntimeError(f"Hadoop job failed with exit code {result.returncode}")

parts = sorted(OUTPUT.glob("part-*"))
assert parts, "Hadoop completed but produced no part files"

columns = [
    "year", "zone_lat", "zone_lon", "mean_lst_c", "mean_ndvi",
    "record_count", "min_lst_c", "max_lst_c", "latitude", "longitude"
]
summary = pd.concat(
    [pd.read_csv(p, sep="\t", header=None, names=columns) for p in parts],
    ignore_index=True
)

assert not summary.empty, "Output summary is empty"
assert summary["year"].isin([2022, 2023, 2024, 2025]).all()
assert summary["record_count"].sum() == len(df), (
    "Input/output record counts do not match; inspect the Hadoop logs."
)

summary.to_csv(BASE / "uhi_zonal_summary.csv", index=False)

annual = summary.groupby("year").apply(
    lambda g: pd.Series({
        "mean_lst_c": (g["mean_lst_c"] * g["record_count"]).sum()
                      / g["record_count"].sum(),
        "mean_ndvi": (g["mean_ndvi"] * g["record_count"]).sum()
                     / g["record_count"].sum(),
        "grid_records": int(g["record_count"].sum()),
        "hottest_zone_mean_c": g["mean_lst_c"].max(),
        "coolest_zone_mean_c": g["mean_lst_c"].min(),
    }),
    include_groups=False
).reset_index()

annual.to_csv(BASE / "uhi_annual_summary.csv", index=False)

print("\nHADOOP JOB SUCCESSFUL")
print("Input records:", len(df))
print("Geographic zones across all years:", len(summary))
print("Output files:")
print(BASE / "uhi_zonal_summary.csv")
print(BASE / "uhi_annual_summary.csv")
display(annual)
display(summary.head())


Starting Hadoop Streaming job...

HADOOP JOB SUCCESSFUL
Input records: 95436
Geographic zones across all years: 7544
Output files:
/content/uhi_zonal_summary.csv
/content/uhi_annual_summary.csv


,year,mean_lst_c,mean_ndvi,grid_records,hottest_zone_mean_c,coolest_zone_mean_c
0,2022,43.241995,0.299903,23859.0,51.236,30.002
1,2023,42.912337,0.357548,23859.0,49.842,29.181
2,2024,42.267646,0.300485,23859.0,50.191,29.067
3,2025,39.978536,0.334007,23859.0,46.123,27.995


,year,zone_lat,zone_lon,mean_lst_c,mean_ndvi,record_count,min_lst_c,max_lst_c,latitude,longitude
0,2022,17.2,78.20,45.006,0.33711,6,42.195,46.783,17.202429,78.201576
1,2022,17.2,78.21,46.824,0.33958,6,45.328,48.185,17.202547,78.210030
2,2022,17.2,78.22,46.582,0.34261,10,44.661,48.261,17.201878,78.220469
3,2022,17.2,78.23,45.528,0.35837,9,43.344,47.188,17.201465,78.229777
4,2022,17.2,78.24,46.556,0.33446,12,43.169,49.495,17.201600,78.239641


In [8]:

from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

destination = Path("/content/drive/MyDrive/Hyderabad_UHI")
destination.mkdir(parents=True, exist_ok=True)

for filename in [
    "hyderabad_heat.csv",
    "mapper.py",
    "reducer.py",
    "uhi_zonal_summary.csv",
    "uhi_annual_summary.csv",
]:
    source = Path("/content") / filename
    if not source.is_file():
        raise FileNotFoundError(f"Missing expected file: {source}")
    shutil.copy2(source, destination / filename)

print("Saved project files to Google Drive:")
print(destination)
for file in sorted(destination.iterdir()):
    print(f"{file.name}: {file.stat().st_size:,} bytes")


Mounted at /content/drive
Saved project files to Google Drive:
/content/drive/MyDrive/Hyderabad_UHI
hyderabad_heat.csv: 9,268,376 bytes
mapper.py: 758 bytes
reducer.py: 1,274 bytes
uhi_annual_summary.csv: 339 bytes
uhi_zonal_summary.csv: 512,597 bytes


In [9]:

from pathlib import Path
import zipfile

project = Path("/content/drive/MyDrive/Hyderabad_UHI")
assert (project / "uhi_zonal_summary.csv").exists()
assert (project / "uhi_annual_summary.csv").exists()

app_code = r'''
from pathlib import Path
import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(
    page_title="Hyderabad Urban Heat Analytics",
    page_icon="🌡️",
    layout="wide"
)

ROOT = Path(__file__).parent

@st.cache_data
def load_data():
    zonal = pd.read_csv(ROOT / "uhi_zonal_summary.csv")
    annual = pd.read_csv(ROOT / "uhi_annual_summary.csv")
    zonal["year"] = zonal["year"].astype(int)
    annual["year"] = annual["year"].astype(int)
    return zonal, annual

try:
    zonal, annual = load_data()
except Exception as e:
    st.error(f"Could not load dashboard data: {e}")
    st.stop()

st.title("🌡️ Hyderabad Urban Heat Island Analytics")
st.caption(
    "Satellite-derived land-surface temperature (LST) and "
    "Normalized Difference Vegetation Index (NDVI), 2022–2025"
)

years = sorted(zonal["year"].unique())
selected_year = st.sidebar.selectbox("Analysis year", years, index=len(years)-1)
heat = zonal[zonal["year"] == selected_year].copy()

st.sidebar.markdown("---")
st.sidebar.caption("LST is land-surface temperature, not air temperature.")
st.sidebar.caption("Zone averages are grouped using rounded geographic coordinates.")

c1, c2, c3, c4 = st.columns(4)
c1.metric("Year", str(selected_year))
c2.metric("Mean LST", f"{heat['mean_lst_c'].mean():.2f} °C")
c3.metric("Mean NDVI", f"{heat['mean_ndvi'].mean():.3f}")
c4.metric("Geographic zones", f"{len(heat):,}")

st.subheader(f"Temperature distribution — {selected_year}")
map_fig = px.scatter_map(
    heat,
    lat="latitude",
    lon="longitude",
    color="mean_lst_c",
    size="record_count",
    hover_name="year",
    hover_data={
        "mean_lst_c": ":.2f",
        "mean_ndvi": ":.3f",
        "record_count": True,
        "latitude": ":.4f",
        "longitude": ":.4f"
    },
    color_continuous_scale="Inferno",
    size_max=13,
    zoom=9,
    center={"lat": 17.385, "lon": 78.4867},
    map_style="open-street-map",
    labels={
        "mean_lst_c": "Mean LST (°C)",
        "mean_ndvi": "Mean NDVI",
        "record_count": "Records"
    }
)
map_fig.update_layout(height=600, margin={"l": 0, "r": 0, "t": 10, "b": 0})
st.plotly_chart(map_fig, use_container_width=True)

left, right = st.columns(2)

with left:
    st.subheader("Annual temperature trend")
    trend = px.line(
        annual, x="year", y="mean_lst_c", markers=True,
        labels={"year": "Year", "mean_lst_c": "Mean LST (°C)"}
    )
    trend.update_layout(xaxis=dict(dtick=1))
    st.plotly_chart(trend, use_container_width=True)

with right:
    st.subheader("Vegetation vs. temperature")
    relation = px.scatter(
        heat, x="mean_ndvi", y="mean_lst_c",
        color="mean_lst_c", color_continuous_scale="Inferno",
        labels={"mean_ndvi": "Mean NDVI", "mean_lst_c": "Mean LST (°C)"},
        hover_data=["latitude", "longitude", "record_count"]
    )
    st.plotly_chart(relation, use_container_width=True)

st.subheader("Annual summary")
st.dataframe(annual.round(3), use_container_width=True, hide_index=True)

st.subheader("Zone-level data")
st.dataframe(
    heat.sort_values("mean_lst_c", ascending=False),
    use_container_width=True,
    hide_index=True
)

st.download_button(
    "Download selected year's zone data",
    data=heat.to_csv(index=False).encode("utf-8"),
    file_name=f"hyderabad_uhi_{selected_year}.csv",
    mime="text/csv"
)

st.download_button(
    "Download annual summary",
    data=annual.to_csv(index=False).encode("utf-8"),
    file_name="uhi_annual_summary.csv",
    mime="text/csv"
)

st.caption(
    "Interpretation note: higher LST indicates hotter land surfaces. "
    "NDVI indicates vegetation greenness; correlation does not establish causation."
)
'''

(project / "app.py").write_text(app_code.strip() + "\n")

(project / "requirements.txt").write_text(
    "streamlit>=1.40,<2\n"
    "pandas>=2.0,<3\n"
    "plotly>=5.24,<7\n"
)

(project / "README.md").write_text(
    "# Hyderabad Urban Heat Island Analytics\n\n"
    "Interactive Streamlit dashboard for satellite-derived land-surface "
    "temperature and NDVI summaries for Hyderabad, 2022–2025.\n\n"
    "## Run locally\n\n"
    "```bash\npip install -r requirements.txt\nstreamlit run app.py\n```\n"
)

bundle = Path("/content/Hyderabad_UHI_Dashboard.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for name in [
        "app.py",
        "requirements.txt",
        "README.md",
        "uhi_zonal_summary.csv",
        "uhi_annual_summary.csv",
    ]:
        z.write(project / name, arcname=name)

print("Dashboard source created.")
print("Files:")
for name in ["app.py", "requirements.txt", "README.md",
             "uhi_zonal_summary.csv", "uhi_annual_summary.csv"]:
    print(f"  {name}: {(project / name).stat().st_size:,} bytes")
print("\nBundle:", bundle)
print("Next: download the ZIP using the cell below.")


Dashboard source created.
Files:
  app.py: 3,720 bytes
  requirements.txt: 50 bytes
  README.md: 249 bytes
  uhi_zonal_summary.csv: 512,597 bytes
  uhi_annual_summary.csv: 339 bytes

Bundle: /content/Hyderabad_UHI_Dashboard.zip
Next: download the ZIP using the cell below.


In [10]:

from google.colab import files
files.download("/content/Hyderabad_UHI_Dashboard.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>